In [4]:
import pandas as pd
import numpy as np
import lightgbm as lgb

In [5]:
# 1) Leer todos los archivos y mergearlos en un solo dataset
def cargar_y_combinar_datos():
    # Cargar archivos
    sell_in = pd.read_csv('sell-in.txt', sep='\t')
    stocks = pd.read_csv('tb_stocks.txt', sep='\t')
    productos = pd.read_csv('tb_productos.txt', sep='\t')
    #drop duplicates in productos
    productos = productos.drop_duplicates(subset=['product_id'])
    # drop column description from productos
    productos = productos.drop(columns=['descripcion'], errors='ignore')
    
    # Unir datasets
    df = sell_in.merge(stocks, on=['periodo', 'product_id'], how='left')
    df = df.merge(productos, on='product_id', how='left')
    
    return df

def cargar_datos():
    df = pd.read_csv('sell-in.txt', sep='\t')
    return df

def combinar_datos(df):
    stocks = pd.read_csv('tb_stocks.txt', sep='\t')
    productos = pd.read_csv('tb_productos.txt', sep='\t')
    #drop duplicates in productos
    productos = productos.drop_duplicates(subset=['product_id'])
    # drop column description from productos
    productos = productos.drop(columns=['descripcion'], errors='ignore')

    # Unir datasets
    stocks = transformar_periodo(stocks)
    stocks['fecha'] = stocks['fecha'].astype('period[M]')
    df = df.merge(stocks[["stock_final", 'fecha', 'product_id']], on=['fecha', 'product_id'], how='left')
    df = df.merge(productos, on='product_id', how='left')
    
    return df


# 2) Transformar periodo en date
def transformar_periodo(df):
    df['fecha'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')
    return df


# 3) Rellenar datos faltantes para series temporales
def completar_series_temporales(df):
    import pandas as pd

    # Asegurar que 'fecha' es Period[M]
    df['fecha'] = df['fecha'].astype('period[M]')
    # ordeno por fecha
    df = df.sort_values(by=['product_id', 'customer_id', 'fecha'])

    columnas_forward_fill = ['plan_precios_cuidados']
    columnas_a_rellenar = ['cust_request_qty', 'cust_request_tn', 'tn']

    ## 1. Obtener todos los valores únicos de producto, cliente y rango de fechas completo
    all_product_customer = df[['product_id', 'customer_id']].drop_duplicates()
    fecha_min = df['fecha'].min()
    fecha_max = df['fecha'].max()
    todas_fechas = pd.period_range(start=fecha_min, end=fecha_max, freq='M')

    # 2. Crear el cartesian product de (product_id, customer_id, fecha)
    full_index = (
        all_product_customer
        .assign(key=1)
        .merge(pd.DataFrame({'fecha': todas_fechas, 'key': 1}), on='key')
        .drop('key', axis=1)
    )

    # 3. Merge con el dataframe original para obtener datos completos
    df_full = full_index.merge(df, on=['product_id', 'customer_id', 'fecha'], how='left')

    # 4. Ordenar correctamente
    df_full = df_full.sort_values(['product_id', 'customer_id', 'fecha'])

    # 5. Forward fill para columnas deseadas (por grupo)
    df_full[columnas_forward_fill] = (
        df_full
        .groupby(['product_id', 'customer_id'])[columnas_forward_fill]
        .ffill()
    )

    # 6. Rellenar valores faltantes de demanda con 0
    df_full[columnas_a_rellenar] = df_full[columnas_a_rellenar].fillna(0)

    # drop rows where precios_cuidados es nan porque significa que son fechas que no existia ese cliente
    df_full = df_full.dropna(subset=['plan_precios_cuidados'])

    return df_full.reset_index(drop=True)


#4) Crear variable objetivo (t+2)
def crear_variable_objetivo(df):
    # Ordenamos por producto, cliente y fecha
    df = df.sort_values(['product_id', 'customer_id', 'fecha'])
    
    ## Creamos un dataframe para el shift
    #df_shift = df[['product_id', 'customer_id', 'fecha', 'tn']].copy()
    
    # Shift de 2 meses para el target
    #df_shift['fecha_target'] = df_shift['fecha'] + pd.DateOffset(months=2)
    #df_shift.rename(columns={'tn': 'target', 'fecha': 'fecha_origen'}, inplace=True)
    
    ## Unimos con el dataframe original
    #df = df.merge(df_shift[['product_id', 'customer_id', 'fecha_target', 'target']], 
    #             left_on=['product_id', 'customer_id', 'fecha'], 
    #             right_on=['product_id', 'customer_id', 'fecha_target'], 
    #             how='left')
    # el codigo de arriba esta mal, hay que hacer el shift 2 (ya que esta rellenado con 0s), fecha_target no es necesario
    df['target'] = df.groupby(['product_id', 'customer_id'])['tn'].shift(-2)

    return df


# 5) Crear features para el modelo
def crear_features(df, lag_columns=["tn", "cust_request_qty"]):
    # Extraer mes del año
    df['mes'] = df['fecha'].dt.month
    df["year"] = df["fecha"].dt.year
    

    # ratio between cust_request_qty and cust_request_tn (fill Na with 0)
    df["cust_request_ratio"] = df["cust_request_qty"] / df["cust_request_tn"]
    df["cust_request_ratio"] = df["cust_request_ratio"].fillna(0)

    # Crear lags (valores anteriores)
    for i in [1,2,6,12]:  # Lags de 1 a 6 meses
        print(f"Creando lag {i}")
        for col in lag_columns:
            df[f'{col}_lag_{i}'] = df.groupby(['product_id', 'customer_id'])[col].shift(i)
    
    # Crear medias móviles
    for n in [3, 6]:  # Medias de 3, 6 y 12 meses
        print(f"Creando media {n}")
        for col in lag_columns:
            df[f'{col}_media_{n}'] = df.groupby(['product_id', 'customer_id'])[col].transform(
                lambda x: x.rolling(window=n, min_periods=n).mean()
            )
    
    # Crear máximos y mínimos móviles
    for n in [3]:
        for col in lag_columns:
            print(f"Creando max {n}")
            df[f'{col}_max_{n}'] = df.groupby(['product_id', 'customer_id'])[col].transform(
                lambda x: x.rolling(window=n, min_periods=n).max()
            )
            print(f"Creando min {n}")
            df[f'{col}_min_{n}'] = df.groupby(['product_id', 'customer_id'])[col].transform(
                lambda x: x.rolling(window=n, min_periods=n).min()
            )
    
    # Tendencia (diferencia entre media de 3 y 6 meses)
    for col in lag_columns:
        df[f'{col}_tendencia'] = df[f'{col}_media_3'] - df[f'{col}_media_6']
    
    for col in lag_columns:
        df["stock_ratio"] = df["stock_final"] / (df[f"{col}_media_3"] + 1)  # +1 para evitar división por cero
    
    return df 


#6) Separar el dataset
def separar_dataset(df):
    fechas_ordenadas = sorted(df['fecha'].unique())
    ultima_fecha = fechas_ordenadas[-1]
    penultima_fecha = fechas_ordenadas[-3]
    antepenultima_fecha = fechas_ordenadas[-4]
    
    # Dataset para predicción final (Kaggle)
    kaggle_pred = df[df['fecha'] == ultima_fecha].copy()
    
    # Dataset de test
    test = df[df['fecha'] == penultima_fecha].copy()
    
    # Dataset de evaluación
    eval_data = df[df['fecha'] == antepenultima_fecha].copy()
    
    # Dataset de entrenamiento
    train = df[(df['fecha'] < antepenultima_fecha) & (df['fecha'] != ultima_fecha)].copy()
    
    return train, eval_data, test, kaggle_pred



# Creamos una clase para la métrica personalizada que necesita acceso a product_id
class CustomMetric:
    def __init__(self, df_eval, product_id_col='product_id'):
        self.df_eval = df_eval
        self.product_id_col = product_id_col
    
    def __call__(self, preds, train_data):
        labels = train_data.get_label()
        df_temp = self.df_eval.copy()
        df_temp['preds'] = preds
        df_temp['labels'] = labels
        
        # Agrupar por product_id y calcular el error
        por_producto = df_temp.groupby(self.product_id_col).agg({'labels': 'sum', 'preds': 'sum'})
        
        # Calcular el error personalizado
        error = np.sum(np.abs(por_producto['labels'] - por_producto['preds'])) / np.sum(por_producto['labels'])
        
        # LightGBM espera que el segundo valor sea mayor cuando el modelo es mejor
        return 'custom_error', error, False

def entrenar_modelo(X_train, y_train, X_eval, y_eval, df_eval):
    cat_features = [col for col in X_train.columns if X_train[col].dtype.name == 'category']
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
    eval_data = lgb.Dataset(X_eval, label=y_eval, reference=train_data, categorical_feature=cat_features)
    
    df_eval_metric = df_eval[['product_id']].copy()
    custom_metric = CustomMetric(df_eval_metric)
    
    params = {
        'objective': 'regression',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_data_in_leaf': 20,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1
    }
    
    callbacks = [
        lgb.early_stopping(50),
        lgb.log_evaluation(100)
    ]
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[eval_data],
        feval=custom_metric,
        callbacks=callbacks
    )
    return model
    
# 8) Evaluar el modelo en test
def evaluar_modelo(model, X_test, test_df):
    # Predicciones
    predictions = model.predict(X_test)
    test_df['predictions'] = predictions
    
    # Error por producto
    product_actual = test_df.groupby('product_id')['target'].sum()
    product_pred = test_df.groupby('product_id')['predictions'].sum()
    
    # Crear DataFrame de evaluación
    eval_df = pd.DataFrame({
        'product_id': product_actual.index,
        'tn_real': product_actual.values,
        'tn_predicha': product_pred.values
    })
    
    # Calcular el error personalizado
    total_error = np.sum(np.abs(eval_df['tn_real'] - eval_df['tn_predicha'])) / np.sum(eval_df['tn_real'])
    
    print(f"Error en test: {total_error:.4f}")
    print("\nTop 5 productos con mayor error absoluto:")
    eval_df['error_absoluto'] = np.abs(eval_df['tn_real'] - eval_df['tn_predicha'])
    print(eval_df.sort_values('error_absoluto', ascending=False).head())
    
    return eval_df, total_error


# Función para optimizar el dataframe y reducir memoria
def optimizar_dataframe(df):
    # Copia para no modificar el original
    df_optimizado = df.copy()
    
    # Convertir float64 a float32
    for col in df_optimizado.select_dtypes(include=['float64']).columns:
        df_optimizado[col] = df_optimizado[col].astype('float32')
    
    # Optimizar fecha - usar periodo_date con solo mes y año
    if 'fecha' in df_optimizado.columns:
        # Usar un formato más ligero para fecha (solo mes y año)
        df_optimizado['fecha'] = pd.PeriodIndex(df_optimizado['fecha'], freq='M')
    
    # Nota: Mantener 'periodo' ya que se usa para operaciones posteriores
    # pero se podría eliminar al final del proceso si ya no es necesario
    
    # Convertir customer_id y product_id a int32
    if 'customer_id' in df_optimizado.columns:
        df_optimizado['customer_id'] = df_optimizado['customer_id'].astype("uint32")
    
    if 'product_id' in df_optimizado.columns:
        df_optimizado['product_id'] = df_optimizado['product_id'].astype("uint32")
    
    # Convertir plan_precios_cuidados a categoría
    if 'plan_precios_cuidados' in df_optimizado.columns:
        df_optimizado['plan_precios_cuidados'] = df_optimizado['plan_precios_cuidados'].astype('category')
    
    # Convertir cust_request_qty a int32 si es posible
    if 'cust_request_qty' in df_optimizado.columns:
        if df_optimizado['cust_request_qty'].dropna().apply(lambda x: float(x).is_integer()).all():
            df_optimizado['cust_request_qty'] = df_optimizado['cust_request_qty'].fillna(0).astype('int32')
        else:
            df_optimizado['cust_request_qty'] = df_optimizado['cust_request_qty'].astype('float32')
    
    # Convertir columnas de objeto a categorías
    for col in df_optimizado.select_dtypes(include=['object']).columns:
        df_optimizado[col] = df_optimizado[col].astype('category')
    
    # Mostrar información sobre la reducción de memoria
    print(f"Memoria antes de optimizar: {df.memory_usage().sum() / 1024**2:.2f} MB")
    print(f"Memoria después de optimizar: {df_optimizado.memory_usage().sum() / 1024**2:.2f} MB")
    
    return df_optimizado


def completar_series_temporales_v1(df):
    """
    Esta version completa las series temporales de productos y clientes cuando AMBOS estaban activos. 
    Entiendo ACTIVOS como los periodos que son igual o mayores a la primer fecha de venta de cada cliente y producto.
    """
    import pandas as pd

    # Asegurar que 'fecha' es Period[M]
    df['fecha'] = df['fecha'].astype('period[M]')

    columnas_forward_fill = ['plan_precios_cuidados', 'stock_final', 'cat1', 'cat2', 'cat3', 'brand', 'sku_size']
    columnas_a_rellenar = ['cust_request_qty', 'cust_request_tn', 'tn']

    # 1. Fecha mínima de aparición por cliente y producto
    fecha_ini_clientes = df.groupby('customer_id')['fecha'].min().reset_index().rename(columns={'fecha': 'fecha_ini_c'})
    fecha_ini_productos = df.groupby('product_id')['fecha'].min().reset_index().rename(columns={'fecha': 'fecha_ini_p'})

    # 2. Rango completo de fechas
    fechas = pd.period_range(df['fecha'].min(), df['fecha'].max(), freq='M')
    fechas_df = pd.DataFrame({'fecha': fechas})

    # 3. Combinar clientes x fechas >= fecha_ini
    clientes_fechas = fecha_ini_clientes.merge(fechas_df, how='cross')
    clientes_fechas = clientes_fechas[clientes_fechas['fecha'] >= clientes_fechas['fecha_ini_c']]
    clientes_fechas = clientes_fechas[['customer_id', 'fecha']]

    productos_fechas = fecha_ini_productos.merge(fechas_df, how='cross')
    productos_fechas = productos_fechas[productos_fechas['fecha'] >= productos_fechas['fecha_ini_p']]
    productos_fechas = productos_fechas[['product_id', 'fecha']]

    # 4. Intersección cliente-producto donde ambos ya eran activos (pero sin cortar por fecha final)
    posibles_combinaciones = productos_fechas.merge(clientes_fechas, on='fecha', how='inner')

    # 5. Merge con datos originales
    df_full = posibles_combinaciones.merge(df, on=['product_id', 'customer_id', 'fecha'], how='left')

    # 6. Ordenar
    df_full = df_full.sort_values(['product_id', 'customer_id', 'fecha'])

    # 7. Forward fill por grupo
    df_full[columnas_forward_fill] = (
        df_full.groupby(['product_id', 'customer_id'])[columnas_forward_fill].ffill()
    )

    # 8. Completar demanda con 0
    df_full[columnas_a_rellenar] = df_full[columnas_a_rellenar].fillna(0)

    # dropea las rows donde el el producto no existia
    stocks = pd.read_csv('tb_stocks.txt', sep='\t')
    productos = pd.read_csv('tb_productos.txt', sep='\t')
    #drop duplicates in productos
    productos = productos.drop_duplicates(subset=['product_id'])
    productos = productos.drop(columns=['descripcion'], errors='ignore')

    # Unir datasets
    # drop stock_final y cat1	cat2	cat3	brand	sku_size	ya que se hacen en el merge
    df_full = df_full.drop(columns=['stock_final', 'cat1', 'cat2', 'cat3', 'brand', 'sku_size'])
    df_full = df_full.merge(stocks, on=['periodo', 'product_id'], how='left')
    df_full = df_full.merge(productos, on='product_id', how='left')
    
    return df_full.reset_index(drop=True)

def completar_series_temporales_v2(df):
    """
    Esta versión completa las series temporales de productos y clientes cuando AMBOS estaban activos.
    Se considera 'ACTIVO' el periodo entre la primera y última fecha de venta de cada cliente y producto.
    """
    import pandas as pd

    # Asegurar que 'fecha' es Period[M]
    df['fecha'] = pd.to_datetime(df['periodo'].astype(str), format='%Y%m')
    df['fecha'] = df['fecha'].astype('period[M]')

    columnas_forward_fill = ['plan_precios_cuidados']
    columnas_a_rellenar = ['cust_request_qty', 'cust_request_tn', 'tn', 'plan_precios_cuidados']

    # 1. Fecha mínima y máxima de aparición por cliente y producto
    fechas_clientes = df.groupby('customer_id')['fecha'].agg(fecha_ini_c='min', fecha_fin_c='max').reset_index()
    fechas_productos = df.groupby('product_id')['fecha'].agg(fecha_ini_p='min', fecha_fin_p='max').reset_index()

    # los clientes donde la fecha de fin sea menor a 2019-09 lo paso a 2020-01
    fechas_clientes.loc[fechas_clientes['fecha_fin_c'] >= '2019-09', 'fecha_fin_c'] = '2020-01'
    fechas_productos.loc[fechas_productos['fecha_fin_p'] >= '2019-09', 'fecha_fin_p'] = '2020-01'

    # si la primer compra del cliente es en 2017-03, o 2017-02 lo paso a 2017-01
    fechas_clientes.loc[fechas_clientes['fecha_ini_c'] <= '2017-03', 'fecha_ini_c'] = '2017-01'
    fechas_productos.loc[fechas_productos['fecha_ini_p'] <= '2017-03', 'fecha_ini_p'] = '2017-01'

    # print number of fechas that are 2020-01
    print(f"Número de clientes con fecha_fin_c en 2020-01: {fechas_clientes[fechas_clientes['fecha_fin_c'] == '2020-01'].shape[0]}")
    print(f"Número de clientes con fecha_ini_c en 2017-01: {fechas_clientes[fechas_clientes['fecha_ini_c'] == '2017-01'].shape[0]}")

    # 2. Rango completo de fechas
    fechas = pd.period_range(df['fecha'].min(), df['fecha'].max(), freq='M')
    fechas_df = pd.DataFrame({'fecha': fechas})

    # 3. Crear combinación cliente-fechas activas
    clientes_fechas = fechas_clientes.merge(fechas_df, how='cross')
    clientes_fechas = clientes_fechas[

        (clientes_fechas['fecha'] >= clientes_fechas['fecha_ini_c']) &
        (clientes_fechas['fecha'] <= clientes_fechas['fecha_fin_c'])
    ][['customer_id', 'fecha']]

    # 4. Crear combinación producto-fechas activas
    productos_fechas = fechas_productos.merge(fechas_df, how='cross')
    productos_fechas = productos_fechas[
        (productos_fechas['fecha'] >= productos_fechas['fecha_ini_p']) &
        (productos_fechas['fecha'] <= productos_fechas['fecha_fin_p'])
    ][['product_id', 'fecha']]

    # 5. Intersección cliente-producto donde ambos eran activos
    posibles_combinaciones = productos_fechas.merge(clientes_fechas, on='fecha', how='inner')

    # 6. Merge con datos originales
    df_full = posibles_combinaciones.merge(df, on=['product_id', 'customer_id', 'fecha'], how='left')

    # 7. Ordenar
    df_full = df_full.sort_values(['product_id', 'customer_id', 'fecha'])

    # 8. Forward fill por grupo
    #df_full[columnas_forward_fill] = (
    #    df_full.groupby(['product_id', 'customer_id'])[columnas_forward_fill].ffill()
    #)

    # 9. Completar demanda con 0
    df_full[columnas_a_rellenar] = df_full[columnas_a_rellenar].fillna(0)

    return df_full.reset_index(drop=True)

In [6]:
df = cargar_datos()
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'sell-in.txt'

In [ ]:
df = transformar_periodo(df)
df.head()

,periodo,customer_id,product_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,fecha
0,201701,10234,20524,0,2,0.05300,0.05300,2017-01-01
1,201701,10032,20524,0,1,0.13628,0.13628,2017-01-01
2,201701,10217,20524,0,1,0.03028,0.03028,2017-01-01
3,201701,10125,20524,0,1,0.02271,0.02271,2017-01-01
4,201701,10012,20524,0,11,1.54452,1.54452,2017-01-01


In [ ]:
product_ids = df['product_id'].unique()
customer_ids = df['customer_id'].unique()
periodos = pd.date_range(
    start=df['fecha'].min(),
    end=df['fecha'].max(),
    freq="MS"
)
cartesian = pd.MultiIndex.from_product(
    [product_ids, customer_ids, periodos],
    names=['product_id', 'customer_id', 'fecha']
).to_frame(index=False)
periodo_producto = df.groupby("product_id")["fecha"].agg(["min", "max"]).reset_index()
periodo_producto.columns = ["product_id", "periodo_min_producto", "periodo_max_producto"]

periodo_customer = df.groupby("customer_id")["fecha"].agg(["min", "max"]).reset_index()
periodo_customer.columns = ["customer_id", "periodo_min_customer", "periodo_max_customer"]

cartesian = cartesian.merge(periodo_producto, on="product_id", how="left")
cartesian = cartesian.merge(periodo_customer, on="customer_id", how="left")

cartesian = cartesian[
    (cartesian["fecha"] >= cartesian["periodo_min_producto"]) &
    (cartesian["fecha"] <= cartesian["periodo_max_producto"]) &
    (cartesian["fecha"] >= cartesian["periodo_min_customer"])
].copy()

df_final = cartesian.merge(df, on=["product_id", "customer_id", "fecha"], how="left")
df_final.fillna(0, inplace=True)
df_final['periodo'] = df_final['fecha'].dt.strftime('%Y%m').astype(int)


In [ ]:
# le agrego la columna periodo que es AAAAMM
df_final = optimizar_dataframe(df_final)

Memoria antes de optimizar: 1572.28 MB
Memoria después de optimizar: 1130.07 MB


In [ ]:
stocks = pd.read_csv('tb_stocks.txt', sep='\t')
productos = pd.read_csv('tb_productos.txt', sep='\t')
#drop duplicates in productos
productos = productos.drop_duplicates(subset=['product_id'])
productos = productos.drop(columns=['descripcion'], errors='ignore')

# Unir datasets
# drop stock_final y cat1	cat2	cat3	brand	sku_size	ya que se hacen en el merge
df_final = df_final.merge(stocks, on=['periodo', 'product_id'], how='left')
df_final = df_final.merge(productos, on='product_id', how='left')

df_final

,product_id,customer_id,fecha,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,periodo,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,cat3,brand,sku_size
0,20524,10234,2017-01,2017-01-01,2019-12-01,2017-01-01,2019-12-01,201701,0.0,2,0.05300,0.05300,NaN,HC,VAJILLA,Cristalino,Importado,500.0
1,20524,10234,2017-02,2017-01-01,2019-12-01,2017-01-01,2019-12-01,201702,0.0,0,0.00000,0.00000,NaN,HC,VAJILLA,Cristalino,Importado,500.0
2,20524,10234,2017-03,2017-01-01,2019-12-01,2017-01-01,2019-12-01,201703,0.0,1,0.01514,0.01514,NaN,HC,VAJILLA,Cristalino,Importado,500.0
3,20524,10234,2017-04,2017-01-01,2019-12-01,2017-01-01,2019-12-01,201704,0.0,0,0.00000,0.00000,NaN,HC,VAJILLA,Cristalino,Importado,500.0
4,20524,10234,2017-05,2017-01-01,2019-12-01,2017-01-01,2019-12-01,201705,0.0,0,0.00000,0.00000,NaN,HC,VAJILLA,Cristalino,Importado,500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17173443,20770,10591,2019-12,2019-12-01,2019-12-01,2019-11-01,2019-11-01,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0
17173444,20770,10559,2019-12,2019-12-01,2019-12-01,2019-11-01,2019-11-01,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0
17173445,20770,10560,2019-12,2019-12-01,2019-12-01,2019-11-01,2019-12-01,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0
17173446,20770,10582,2019-12,2019-12-01,2019-12-01,2019-12-01,2019-12-01,201912,0.0,0,0.00000,0.00000,9.53225,HC,PROFESIONAL,LV ROPA POLVO,INDUSTRIAL,25.0


In [ ]:
df.drop(columns=["periodo_min_producto", "periodo_max_producto", "periodo_min_customer", "periodo_max_customer"], inplace=True, errors='ignore')

In [ ]:
df_final.to_parquet('df_intermedio.parquet')